# Indian Courtroom Judge AI - Vanilla PEFT QLoRA (No Unsloth)

**Base Model:** meta-llama/Meta-Llama-3-8B-Instruct (4-bit QLoRA via PEFT + bitsandbytes)
**Dataset:** 100K Indian courtroom objections with structured JSON output
**VRAM Target:** 6GB+ (RTX 4050, RTX 3060, GTX 1650, etc.)

This notebook uses **standard Hugging Face libraries** (Transformers, PEFT, TRL, bitsandbytes) without Unsloth.
This ensures maximum compatibility across Windows, Linux, and macOS.

---

## Contents
1. Environment Setup & Windows Troubleshooting
2. VRAM Profiling
3. Auto VRAM-Aware Configuration
4. Dataset Loading & Optional Subsampling
5. Model Loading (Standard 4-bit QLoRA)
6. LoRA Configuration (Standard PEFT)
7. Training (Standard SFTTrainer)
8. Inference & Evaluation
9. Save & Push to Hub
10. FAQ: RTX 4050 / Windows Issues

---

## 1. Environment Setup & Windows Troubleshooting

### Recommended: Use a fresh conda environment

```bash
# Windows (Anaconda Prompt or PowerShell)
conda create -n courtroom python=3.10 -y
conda activate courtroom

# Install PyTorch with CUDA 12.1 (check your CUDA version first)
# For CUDA 11.8: https://pytorch.org/get-started/previous-versions/
pip install torch==2.4.0 torchvision==0.19.0 torchaudio==2.4.0 --index-url https://download.pytorch.org/whl/cu121

# Install transformers, PEFT, TRL, datasets, bitsandbytes
pip install transformers==4.46.2 peft==0.14.0 trl==0.13.0 datasets==3.2.0 bitsandbytes==0.45.0 accelerate==1.2.0 huggingface-hub==0.27.0

# Optional: for jsonl dataset creation
pip install pandas
```

### If you get "CUDA out of memory" even before training:
- Close all browser tabs, Discord, etc.
- In Task Manager, check "GPU - Dedicated GPU Memory" column
- If Windows uses >1GB for desktop, you only have ~4-5GB free
- Set `batch_size=1`, `seq_len=512`, and `gradient_accumulation_steps=16`

### If you get "No module named 'unsloth'" errors:
- This notebook does NOT use Unsloth. The error comes from leftover imports.
- Make sure you're in the fresh conda environment.


In [ ]:
# Core libraries (run this if not already installed)
# !pip install transformers peft trl datasets bitsandbytes accelerate huggingface-hub

import os
import gc
import random
import json
import torch
import numpy as np
from datasets import load_dataset, Dataset
from transformers import (
    AutoModelForCausalLM, AutoTokenizer,
    BitsAndBytesConfig, TrainingArguments
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

SEED = 3407
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")


## 2. VRAM Profiling

In [ ]:
def get_gpu_info():
    if not torch.cuda.is_available():
        print("CUDA not available. Training will fail.")
        return None
    device = torch.cuda.current_device()
    props = torch.cuda.get_device_properties(device)
    total = props.total_memory / 1024**3
    allocated = torch.cuda.memory_allocated(device) / 1024**3
    reserved = torch.cuda.memory_reserved(device) / 1024**3
    free = total - reserved
    print(f"GPU: {props.name}")
    print(f"Total VRAM: {total:.1f} GB")
    print(f"Allocated: {allocated:.1f} GB")
    print(f"Reserved: {reserved:.1f} GB")
    print(f"Free (usable): {free:.1f} GB")
    return {"name": props.name, "total_gb": total, "free_gb": free}

def clear_cache():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

gpu_info = get_gpu_info()

## 3. Auto VRAM-Aware Configuration

In [ ]:
def auto_config(gpu_info):
    free_gb = gpu_info["free_gb"] if gpu_info else 6.0
    if free_gb >= 20:
        return {"max_seq_length": 4096, "batch_size": 2, "accum": 4, "r": 32, "alpha": 32, "bf16": True, "notes": "24GB tier"}
    elif free_gb >= 14:
        return {"max_seq_length": 2048, "batch_size": 1, "accum": 8, "r": 16, "alpha": 16, "bf16": True, "notes": "14GB tier"}
    elif free_gb >= 10:
        return {"max_seq_length": 2048, "batch_size": 1, "accum": 8, "r": 8, "alpha": 8, "bf16": False, "notes": "10GB tier"}
    elif free_gb >= 6:
        return {"max_seq_length": 1024, "batch_size": 1, "accum": 8, "r": 8, "alpha": 8, "bf16": False, "notes": "6GB tier (RTX 4050)"}
    else:
        return {"max_seq_length": 512, "batch_size": 1, "accum": 16, "r": 4, "alpha": 4, "bf16": False, "notes": "4GB tier - extreme limits"}

PREFER_SPEED = True
cfg = auto_config(gpu_info)
print(f"Config: {cfg['notes']}")
for k, v in cfg.items():
    if k != "notes":
        print(f"  {k}: {v}")

MAX_SEQ_LENGTH = cfg['max_seq_length']
BATCH_SIZE = cfg['batch_size']
GRAD_ACCUM = cfg['accum']
LORA_R = cfg['r']
LORA_ALPHA = cfg['alpha']
USE_BF16 = cfg['bf16']

## 4. Dataset Loading & Optional Subsampling

In [ ]:
DATASET_NAME = "pkheria/indian-courtroom-objections-100k"
dataset = load_dataset(DATASET_NAME)

print(f"Train: {len(dataset['train'])} | Test: {len(dataset['test'])}")

# Optional subsampling
DATASET_PERCENT = 0.5  # Set to 1.0 for full, 0.5 for 50%

if DATASET_PERCENT < 1.0:
    train_indices = list(range(len(dataset['train'])))
    random.shuffle(train_indices)
    subset_size = int(len(train_indices) * DATASET_PERCENT)
    keep = sorted(train_indices[:subset_size])
    dataset['train'] = dataset['train'].select(keep)
    print(f"Subsampled to {DATASET_PERCENT*100:.0f}%: {len(dataset['train'])} training examples")
else:
    print("Using FULL dataset (100%)")

# Inspect sample
sample = dataset['train'][0]
for msg in sample['messages']:
    print(f"[{msg['role']}] {len(msg['content'])} chars")

parsed = json.loads(sample['messages'][2]['content'])
print(f"JSON keys: {list(parsed.keys())}")

## 5. Model Loading (Standard 4-bit QLoRA)

Load with BitsAndBytesConfig for 4-bit quantization.
This is the standard Hugging Face approach - no Unsloth required.

In [ ]:
MODEL_NAME = "meta-llama/Meta-Llama-3-8B-Instruct"

clear_cache()
pre = get_gpu_info()

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,      # Nested quantization for more memory savings
    bnb_4bit_quant_type="nf4",             # Normalized float 4-bit
    bnb_4bit_compute_dtype=torch.bfloat16 if USE_BF16 else torch.float16,
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"  # Required for training

# Load model
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",                     # Auto-assign layers to GPU/CPU
    trust_remote_code=True,
    torch_dtype=torch.bfloat16 if USE_BF16 else torch.float16,
)

# Prepare model for k-bit training (required for 4-bit + LoRA)
model = prepare_model_for_kbit_training(model)

post = get_gpu_info()
print(f"Model loaded. VRAM used: ~{post['allocated_gb'] - pre['allocated_gb']:.1f} GB")
print(f"Remaining free: ~{post['free_gb']:.1f} GB")

## 6. LoRA Configuration (Standard PEFT)

In [ ]:
peft_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout=0.05,          # Slight regularization
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()
get_gpu_info()

## 7. Training (Standard SFTTrainer)

Key differences from Unsloth:
- `packing=False` is default (safer for structured JSON)
- `gradient_checkpointing=True` trades speed for memory
- `optim="paged_adamw_8bit"` for low VRAM systems

In [ ]:
OUTPUT_DIR = "./judge_lora_vanilla"

# For very low VRAM (<6GB), use paged_adamw_8bit
# For 6GB+, adamw_torch is fine
optimizer_name = "paged_adamw_8bit" if (gpu_info and gpu_info["free_gb"] < 7) else "adamw_torch"

# Enable gradient checkpointing for memory savings
model.gradient_checkpointing_enable()
model.enable_input_require_grads()  # Required for gradient checkpointing with frozen base

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    max_seq_length=MAX_SEQ_LENGTH,
    num_train_epochs=3,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    warmup_ratio=0.1,
    learning_rate=2e-4,
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    logging_steps=25,
    save_strategy="epoch",
    evaluation_strategy="no",
    bf16=USE_BF16,
    fp16=not USE_BF16,
    seed=SEED,
    report_to="none",
    disable_tqdm=False,
    logging_first_step=True,
    optim=optimizer_name,
    group_by_length=True,       # Minimize padding waste
    max_grad_norm=0.3,          # Gradient clipping for stability
)

eff_batch = BATCH_SIZE * GRAD_ACCUM
steps_per_epoch = len(dataset['train']) // eff_batch
total_steps = steps_per_epoch * 3

print(f"Effective batch: {eff_batch}")
print(f"Steps per epoch: ~{steps_per_epoch}")
print(f"Total steps: ~{total_steps}")
print(f"Optimizer: {optimizer_name}")

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset['train'],
    tokenizer=tokenizer,
)

print("Trainer initialized!")
get_gpu_info()

In [ ]:
# Train with OOM recovery
try:
    trainer_stats = trainer.train()
    print(f"Final loss: {trainer_stats.training_loss:.4f}")
    print(f"Time: {trainer_stats.metrics.get('train_runtime', 0) / 3600:.1f} hours")
    get_gpu_info()
except RuntimeError as e:
    if "out of memory" in str(e).lower():
        print("CUDA OOM! Recovery steps:")
        print("1. Reduce MAX_SEQ_LENGTH to 512")
        print("2. Set optim='paged_adamw_8bit'")
        print("3. Reduce LoRA rank to 4")
        print("4. Restart kernel, re-run from model loading")
        clear_cache()
        raise
    else:
        raise

## 8. Inference & Evaluation

In [ ]:
def parse_ruling_json(text):
    json_start = text.rfind('{')
    json_end = text.rfind('}') + 1
    if json_start != -1 and json_end > json_start:
        try:
            return json.loads(text[json_start:json_end])
        except json.JSONDecodeError:
            pass
    ruling = "SUSTAINED" if "SUSTAINED" in text.upper() else "OVERRULED" if "OVERRULED" in text.upper() else "UNKNOWN"
    return {"ruling": ruling, "reason": text[-200:], "parse_error": True}

# Test single example
test_ex = dataset['test'][0]
input_msgs = test_ex['messages'][:-1]

prompt = tokenizer.apply_chat_template(input_msgs, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt", padding=True, truncation=True, max_length=MAX_SEQ_LENGTH).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=128,
        temperature=0.1,
        top_p=0.95,
        do_sample=True,
        pad_token_id=tokenizer.pad_token_id,
    )

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
parsed = parse_ruling_json(response)
print("=== GROUND TRUTH ===")
print(json.dumps(json.loads(test_ex['messages'][2]['content']), indent=2))
print("\n=== MODEL OUTPUT ===")
print(json.dumps(parsed, indent=2))

## 9. Save & Push to Hub

In [ ]:
# Save LoRA adapter only (small, ~10-50MB)
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"LoRA adapter saved to {OUTPUT_DIR}")

# To merge and save full model (requires more VRAM):
# from peft import AutoPeftModelForCausalLM
# merged_model = model.merge_and_unload()
# merged_model.save_pretrained("./judge_merged")

# Push to Hub
# from huggingface_hub import login
# login(token="YOUR_HF_TOKEN")
# model.push_to_hub("YOUR_USERNAME/indian-courtroom-judge-vanilla")
# tokenizer.push_to_hub("YOUR_USERNAME/indian-courtroom-judge-vanilla")

## 10. FAQ: RTX 4050 / Windows Issues

### "ImportError: No module named 'unsloth'"
This notebook does NOT use Unsloth. If you see this, you're importing from a previous notebook or environment. Check your imports.

### "torch.cuda.OutOfMemoryError"
Windows uses 1-2GB VRAM for desktop compositing. With 6GB total, you may only have 4GB free.
Solutions:
1. Close all other apps (browser, Discord, etc.)
2. Set Windows to "Best Performance" mode
3. Reduce `MAX_SEQ_LENGTH` to 512
4. Set `DATASET_PERCENT = 0.25`
5. Use `optim="paged_adamw_8bit"`

### "BitsAndBytes not installed correctly"
For Windows, bitsandbytes may need manual compilation:
```bash
pip install bitsandbytes-windows
# Or download prebuilt wheels from:
# https://github.com/jllllll/bitsandbytes-windows-wheels
```

### Slow training on laptop
Laptop GPUs have lower TDP (power limit) than desktop cards:
- RTX 4050 Laptop: ~35-50W TDP
- Desktop RTX 3060: ~170W TDP
Expect 3-5x slower training than desktop equivalents. This is normal.

### Can I use Google Colab instead?
Yes! Free Colab gives you a T4 (16GB) which is much faster than RTX 4050 Laptop.
Use the original Unsloth notebooks on Colab - they'll work fine there.